In [1]:
from base_import import *

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import \
                                            ( LinearDiscriminantAnalysis as LDA ,
                                            QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Reading in the Heart Disease Dataset to perform some ML analysis on it

In [10]:
df = pd.read_csv('Data/Heart_disease.csv')

## Splitting data into features and predictors

In [12]:
df.rename(columns = {'TenYearCHD': 'Heart_disease'}, inplace = True)
X = df[['sysBP']]
y = df['Heart_disease']

## Making a Logistic Regression 

In [ ]:
logistic = LogisticRegression()

logistic.fit(X, y)


LogisticRegression()

In [ ]:
#Seeing the coefficient of the logistic regression 
logistic.coef_

array([[0.02405823]])

In [ ]:
#Seeing the intercept of the logistic regression 
logistic.intercept_

array([-5.00016447])

In [5]:
from ISLP import load_data
from ISLP.models import ( ModelSpec as MS , summarize )

In [6]:
from ISLP import confusion_table
from ISLP.models import contrast

## Getting a new feature totChol and testing the inference

In [13]:
X = df['totChol']
y = df['Heart_disease']

In [14]:
# Removing nans in the data
mask = np.isnan(X.values)
y_mask = y[~mask]
X_mask = X[~mask]

In [16]:
X_mask = pd.DataFrame(X_mask)

In [17]:
design = MS(X_mask)
X_mask = design.fit_transform (X_mask)

glm = sm.GLM(y_mask, X_mask, family=sm.families.Binomial())
results = glm.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-2.8942,0.230,-12.607,0.0
totChol,0.0049,0.001,5.268,0.0


## Getting the predictions from the logistic model

Note that due to the way logsitic regression works is that you get a probability and so we need to take the predictions from the model and convert this into a notation where we can compare with the testing set. 

Here we use > 0.5 to be 1 (have heart disease )and < 0.5 to be 0 (no heart disease)

In [19]:
prob = results.predict(X_mask) 
labels = np.zeros(len(X_mask))
labels[prob > 0.5] = 1

# Assessing Model Accuracy using Confusion Matrix/Table

In [20]:
confusion_table(labels ,  y_mask.values).T

Predicted,0,1
Truth,,
0,3554,1
1,634,1


## Checking How another feature heartRate predict heart health

In [21]:
X = df['heartRate']
mask = np.isnan(X.values)
y_mask = y[~mask]
X_mask = X[~mask]
X_mask = pd.DataFrame(X_mask)

#allvars = Smarket.columns.drop (['Today ', 'Direction ', 'Year '])
design = MS(X_mask)
X_mask = design.fit_transform (X_mask)

glm = sm.GLM(y_mask, X_mask, family=sm.families.Binomial())
results = glm.fit()
summarize(results)

prob = results.predict(X_mask) 
labels = np.zeros(len(X_mask))
labels[prob > 0.5] = 1
summarize(results)

,coef,std err,z,P>|z|
intercept,-2.1203,0.272,-7.799,0.000
heartRate,0.0052,0.004,1.491,0.136


In [22]:
confusion_table(labels ,  y_mask.values).T

Predicted,0,1
Truth,,
0,3596,0
1,643,0


In [ ]:
# assessing accuracy 
3596/(643+3596)

0.8483132814343005

# Using Logistic Regression on spam email data

In [23]:
spam_email_df = pd.read_csv('Data/spam_train.csv')

In [24]:
y = spam_email_df['y']
X = spam_email_df.drop('y', axis = 1)

In [26]:
def perform_logistic_regression(X, y, summary = None):

    '''
    Function that performs logistic regression given an input X and predictor y

    Inputs
    ------------------
    X: DataFrame of Features 
    y: Column of predictors
    summary: whether to show the summary of the fit or not. Default is None

    Return
    -------------------

    results: 
    
    '''

    design = MS(X)
    X = design.fit_transform(X)
    glm = sm.GLM(y, 
                 X,
                 family=sm.families.Binomial ())
    results = glm.fit()

    if summary:
        print(summarize(results))

        
    return results

In [27]:
results = perform_logistic_regression(X, y, summary= True)

                              coef  std err       z  P>|z|
intercept                  -2.2381    0.109 -20.549  0.000
word.freq.remove            5.7161    0.560  10.199  0.000
word.freq.order             0.9169    0.213   4.311  0.000
word.freq.free              1.1018    0.139   7.907  0.000
word.freq.meeting          -3.9107    1.094  -3.573  0.000
word.freq.re               -0.3800    0.109  -3.499  0.000
word.freq.edu              -1.2632    0.256  -4.934  0.000
char.freq.semicolon        -1.6941    0.669  -2.531  0.011
char.freq.exclamation       2.2251    0.187  11.901  0.000
capital.run.length.average  0.3421    0.032  10.570  0.000


In [ ]:
#This is the log odds of the word frequency remove
np.exp(5.7161)

np.float64(303.71810954781)

In [ ]:
#This is the log odds of the word frequency semicolon
np.exp(-1.6941 )

np.float64(0.18376454271384293)

In [28]:
prob = results.predict() 
labels = np.zeros(len(X))
labels[prob > 0.5] = 1
confusion_table(labels ,  y).T
#summarize(results)

Predicted,0,1
Truth,,
0,1703,97
1,321,879


In [ ]:
# Accuracy metric
(1703+879)/(1703+879+321+97)

0.8606666666666667

### Altering the input DataFrame by removing non-impactful features

In [29]:
X = spam_email_df.drop(['y', 'word.freq.meeting', 'word.freq.re', 'char.freq.semicolon'], axis = 1)

In [30]:
result_removed = perform_logistic_regression(X, y, summary= True)

                              coef  std err       z  P>|z|
intercept                  -2.4422    0.103 -23.675    0.0
word.freq.remove            6.1240    0.570  10.735    0.0
word.freq.order             1.0906    0.214   5.096    0.0
word.freq.free              1.1440    0.133   8.604    0.0
word.freq.edu              -1.3359    0.261  -5.125    0.0
char.freq.exclamation       2.2407    0.176  12.744    0.0
capital.run.length.average  0.3190    0.030  10.801    0.0


In [31]:
prob = result_removed.predict() 
labels = np.zeros(len(X))
labels[prob > 0.5] = 1
confusion_table(labels ,  y).T
#summarize(results)

Predicted,0,1
Truth,,
0,1696,104
1,338,862


In [ ]:
#Accuracy metric shown here
(1696+862)/(1696+862+  338+  104)

0.8526666666666667

# Recap

In this notebook we covered the basics of logistic regression. We first applied it one variabel ata time to heartdisease data and saw how that changed based off of the input. We also applied it to detecting spam email using some input features and we saw that when using all the columsn we achived an accuracy of 86%. When we removed columns that were not that impactful we say a marginal drop in accuracy going to 85%. 